# 1 - Importa bibliotecas para o uso

In [6]:
import os

import pandas as pd
from churn.cleaning import ColumnMapper, Mappings
from churn.loader import DataLoader
from churn.processor.customer_churn_processor import CustomerChurnProcessor

customer_churn_processor = CustomerChurnProcessor()

# 2 - Importa o dataset e dropa a coluna customerId

In [7]:
df: pd.DataFrame = DataLoader(data_path="../data/raw/dataset.csv").load()
df = df.drop_duplicates(subset="customerID")
df.drop(columns=['customerID'], inplace=True)

# 3 - Converte TotalCharges para número e preenche os valores nulos

Os vazios correspondem a clientes com `tenure` igual a 0, que ainda não foram cobrados, por isso recebem 0.

In [8]:
df = customer_churn_processor.apply_raw_cleaning_data(df)
print(df["TotalCharges"].isna().sum())
df["TotalCharges"] = df["TotalCharges"].fillna(0)

11


# 4 - Mapeia colunas necessárias para o treinamento, aplica one-hot encoding e renomeia o alvo

In [9]:
df = customer_churn_processor.apply_raw_mapping_data(df)
df = ColumnMapper(["Churn"], Mappings.YES_NO).transform(df)
df = customer_churn_processor.apply_raw_categorical_encoding(df)
df.rename(columns={"Churn": "target"}, inplace=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7043 non-null   int64  
 1   SeniorCitizen                            7043 non-null   int64  
 2   MultipleLines                            7043 non-null   int64  
 3   Partner                                  7043 non-null   int64  
 4   Dependents                               7043 non-null   int64  
 5   tenure                                   7043 non-null   int64  
 6   PhoneService                             7043 non-null   int64  
 7   OnlineSecurity                           7043 non-null   int64  
 8   OnlineBackup                             7043 non-null   int64  
 9   DeviceProtection                         7043 non-null   int64  
 10  TechSupport                              7043 non-null   in

# 5 - Validar transformações

In [10]:
total_nulos = df.isna().sum().sum()
colunas_texto = df.select_dtypes(include=["object", "str"]).columns

print(f"Valores nulos: {total_nulos}")
print(f"Colunas de texto restantes: {list(colunas_texto)}")

assert total_nulos == 0, "Ainda há valores nulos no dataset"
assert len(colunas_texto) == 0, "Ainda há colunas de texto no dataset"

Valores nulos: 0
Colunas de texto restantes: []


# 6 - Salva o dataset limpo

In [11]:
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/dataset_clean.csv", index=False)
print(f"Dataset salvo: {df.shape[0]} linhas, {df.shape[1]} colunas")

Dataset salvo: 7043 linhas, 27 colunas
